# Imported Libraries

In [3]:
import pandas as pd
import folium

# Connecting to AWS and pulling Parquet file for Ookla Dataset

In [ ]:
print("Connecting to AWS and pulling Parquet file...")

# The path to the mobile data
s3_path = 's3://ookla-open-data/parquet/performance/type=mobile/year=2023/quarter=4/'

# Read directly from the cloud into your local RAM
mobile_df = pd.read_parquet(s3_path, storage_options={'anon': True})

print("\n--- DATA SECURED ---")
print(f"Total Rows & Columns: {mobile_df.shape}")
print(mobile_df.head())

In [2]:
print("--- Column Names ---")
print(mobile_df.columns.tolist())

print("\n--- First 3 Rows ---")
print(mobile_df.head(3))

--- Column Names ---
['quadkey', 'tile', 'tile_x', 'tile_y', 'avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'avg_lat_down_ms', 'avg_lat_up_ms', 'tests', 'devices']

--- First 3 Rows ---
            quadkey                                               tile  \
0  0022133222330012  POLYGON((-160.037841796875 70.6399478155463, -...   
1  0022133222330013  POLYGON((-160.032348632812 70.6399478155463, -...   
2  0022133222330102  POLYGON((-160.02685546875 70.6399478155463, -1...   

     tile_x  tile_y  avg_d_kbps  avg_u_kbps  avg_lat_ms  avg_lat_down_ms  \
0 -160.0351  70.639       13260        5629          74            879.0   
1 -160.0296  70.639       71946        7040         109           1288.0   
2 -160.0241  70.639       27183        5322          71            919.0   

   avg_lat_up_ms  tests  devices  
0          355.0      1        1  
1          186.0      1        1  
2          203.0      1        1  


# Geo Splicing and kbps to Mbps conversion

In [ ]:
print("\n--- INITIATING GEOSPATIAL SLICER ---")

# Step 1: Define the Bounding Box for the Central Valley
# These coordinates create a box roughly from north of Fresno down past Visalia/Ivanhoe
min_lat = 36.1   # South boundary
max_lat = 36.9   # North boundary
min_lon = -120.0 # West boundary
max_lon = -118.9 # East boundary

# Step 2: The Pandas Filter (Boolean Indexing)
# We use the '&' symbol to combine conditions. 
# We use .copy() at the end to completely sever this new data from the massive 26GB table.
valley_df = mobile_df[
    (mobile_df['tile_y'] >= min_lat) & 
    (mobile_df['tile_y'] <= max_lat) & 
    (mobile_df['tile_x'] >= min_lon) & 
    (mobile_df['tile_x'] <= max_lon)
].copy()

# Step 3: Metric Conversion (kbps to Mbps)
valley_df['avg_d_mbps'] = valley_df['avg_d_kbps'] / 1000
valley_df['avg_u_mbps'] = valley_df['avg_u_kbps'] / 1000

print(f"Global Rows: {len(mobile_df):,}")
print(f"Central Valley Rows: {len(valley_df):,}")

# Step 4: Secure the Payload Locally
# We save this tiny, hyper-focused slice to your actual hard drive.
valley_df.to_parquet('central_valley_mobile_q4.parquet')
print("\n--- LOCAL PAYLOAD SECURED. YOU CAN NOW DROP THE GLOBAL DATA. ---")


--- INITIATING GEOSPATIAL SLICER ---
Global Rows: 3,771,204
Central Valley Rows: 1,884

--- LOCAL PAYLOAD SECURED. YOU CAN NOW DROP THE GLOBAL DATA. ---


In [ ]:
# Because this is local and tiny, this cell will execute in milliseconds
df = pd.read_parquet('central_valley_mobile_q4.parquet')

print(f"Loaded {len(df)} rows perfectly.\n")

# Using .idxmax() and .idxmin() to grab the index of the highest and lowest values
fastest_spot = df.loc[df['avg_d_mbps'].idxmax()]
slowest_spot = df.loc[df['avg_d_mbps'].idxmin()]

print("--- THE GIGABIT TOWER (FASTEST AREA) ---")
print(f"Latitude: {fastest_spot['tile_y']}")
print(f"Longitude: {fastest_spot['tile_x']}")
print(f"Download: {fastest_spot['avg_d_mbps']:.2f} Mbps")
print(f"Tests Run: {fastest_spot['tests']}")

print("\n--- THE DEAD ZONE (SLOWEST AREA) ---")
print(f"Latitude: {slowest_spot['tile_y']}")
print(f"Longitude: {slowest_spot['tile_x']}")
print(f"Download: {slowest_spot['avg_d_mbps']:.2f} Mbps")
print(f"Tests Run: {slowest_spot['tests']}")

Loaded 1884 rows perfectly.

--- THE GIGABIT TOWER (FASTEST AREA) ---
Latitude: 36.886199999999995
Longitude: -119.7592
Download: 1369.75 Mbps
Tests Run: 1

--- THE DEAD ZONE (SLOWEST AREA) ---
Latitude: 36.8159
Longitude: -119.85260000000001
Download: 0.01 Mbps
Tests Run: 1


In [ ]:
robust_df = pd.read_parquet('central_valley_mobile_q4.parquet')

true_fastest = robust_df.loc[robust_df['avg_d_mbps'].idxmax()]
true_slowest = robust_df.loc[robust_df['avg_d_mbps'].idxmin()]

print("--- The True Heavyweight Tower ---")
print(f"Donwload: {true_fastest['avg_d_mbps']:.2f} Mbps (Based on{true_fastest['tests']} tests)")

print("\n--- The True Congested Zone ---")
print(f"Download: {true_slowest['avg_d_mbps']:.2f} Mbps (Based on {true_slowest['tests']} tests)")

# Visualization

map_center = [true_fastest['tile_y'], true_fastest['tile_x']]
valley_map = folium.Map(location=map_center, zoom_start= 11)

folium.Marker(
    location=[true_fastest['tile_y'], true_fastest['tile_x']],
    popup=f"FAST: {true_fastest['avg_d_mbps']:.2f} Mbps",
    icon= folium.Icon(color='green', icon= "arrow-up")
).add_to(valley_map)

folium.Marker(
    location=[true_slowest['tile_y'], true_slowest['tile_x']],
    popup=f"FAST: {true_slowest['avg_d_mbps']:.2f} Mbps",
    icon= folium.Icon(color= "red", icon= "arrow-down")
).add_to(valley_map)

valley_map




--- The True Heavyweight Tower ---
Donwload: 1369.75 Mbps (Based on1 tests)

--- The True Congested Zone ---
Download: 0.01 Mbps (Based on 1 tests)
